# 第三部分：RAII 与资源生命周期

## 实验 2：使用 RAII 管理文件

上一个实验中，`std::FILE*` 的释放责任散落在控制流中。C++ 给出的核心方案是 RAII：让对象拥有资源，把资源生命周期绑定到对象生命周期。

```text
Resource Acquisition → Object Construction
Resource Lifetime    = Object Lifetime
Resource Release     ← Object Destruction
```

本实验实现一个最小文件包装器 `FileGuard`，并观察正常返回、提前返回和打开失败时的行为。重点是所有权与自动清理，完整文件类接口将在后续实验继续扩展。

In [ ]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <cstdio>
#include <filesystem>
#include <iostream>
#include <stdexcept>
#include <string>

### 1. 最小 RAII 包装器

`FileGuard` 对一个 `std::FILE*` 具有唯一所有权：

- 构造函数调用 `std::fopen()` 获取资源；
- 析构函数调用 `std::fclose()` 释放资源；
- `get()` 返回临时借用指针，用于调用 C API；
- 禁止复制，避免两个对象重复关闭同一个文件。

本阶段先不实现移动操作；所有权转移会在后续移动语义阶段展开。

In [ ]:
// 本步骤：通过代码演示“最小 RAII 包装器”并观察结果。
// 定义唯一拥有 FILE* 的最小 RAII 类型。
class FileGuard
{
public:
    FileGuard(
        const char *path,
        const char *mode)
        // 在成员初始化阶段获取句柄并保存用于日志的路径。
        : file_(std::fopen(path, mode)),
          path_(path)
    {
        // 获取失败就抛出异常，避免构造出无效 owner。
        if (file_ == nullptr)
        {
            throw std::runtime_error(
                std::string("cannot open file: ") + path);
        }

        std::cout << "acquire: " << path_ << std::endl;
    }

    // owner 离开作用域时自动关闭句柄，覆盖所有退出路径。
    ~FileGuard()
    {
        std::cout << "release: " << path_ << std::endl;
        std::fclose(file_);
    }

    // 禁止复制，防止两个 owner 对同一 FILE* 重复关闭。
    FileGuard(const FileGuard &) = delete;
    FileGuard &operator=(const FileGuard &) = delete;

    // 临时借出底层句柄；调用方不能关闭或长期保存它。
    std::FILE *get() const
    {
        return file_;
    }

private:
    std::FILE *file_;
    std::string path_;
};

### 2. 构造函数建立有效状态

如果 `std::fopen()` 失败，构造函数抛出异常；如果构造成功，`file_` 就一定有效。于是类建立了一个不变量：

> 每个已经构造完成的 `FileGuard` 都拥有一个有效且尚未关闭的文件句柄。

调用者不再需要创建一个“存在但不可用”的文件对象，也不必在每次使用前重复检查空指针。

### 3. 正常离开作用域

下面向当前实验的 `outputs/02/` 目录写入内容。函数体没有显式调用 `std::fclose()`；局部对象 `file` 在函数返回时自动析构。

In [ ]:
// 本步骤：通过代码演示“正常离开作用域”并观察结果。
// 创建输出目录并集中定义本实验使用的文件路径。
std::filesystem::create_directories("outputs/02");

const char *raii_file_path =
    "outputs/02/output.txt";

// 在函数局部创建 owner，借用句柄完成一次正常写入。
void write_normally()
{
    FileGuard file(raii_file_path, "w");
    std::fputs("written by FileGuard", file.get());
    std::cout << "finish writing" << std::endl;
}

// 函数返回前 FileGuard 已析构，再回到调用方输出日志。
write_normally();
std::cout << "back to caller" << std::endl;

预期顺序是：

```text
acquire
finish writing
release
back to caller
```

`release` 位于函数内部输出和调用方输出之间，说明析构发生在控制流离开 `write_normally()` 的局部作用域时，而不是等到进程结束。

### 4. 提前返回仍会自动释放

函数可以有多个 `return`，但不需要在每个分支重复关闭文件。只要局部对象生命周期结束，析构函数就会执行。

In [ ]:
// 本步骤：通过代码演示“提前返回仍会自动释放”并观察结果。
bool write_with_early_return(bool stop_early)
{
    // 先获取资源并写入第一部分。
    FileGuard file(raii_file_path, "w");
    std::fputs("first part", file.get());

    // 提前返回时不显式清理，依靠局部 owner 自动析构。
    if (stop_early)
    {
        std::cout << "return early" << std::endl;
        return false;
    }

    // 仅正常路径写入第二部分，两条路径都会释放文件。
    std::fputs("second part", file.get());
    return true;
}

// 主动进入提前返回分支，观察 release 仍早于 caller 日志。
write_with_early_return(true);
std::cout << "back to caller" << std::endl;

与实验 1 不同，提前返回只结束 `FileGuard` 对象的作用域，并不会绕过析构函数。释放逻辑仍然只有一份：

```text
任意 return
    ↓
局部对象生命周期结束
    ↓
~FileGuard()
    ↓
std::fclose()
```

### 5. `get()` 返回的是借用指针

`FileGuard` 负责所有权，`get()` 只让现有 C API 临时使用底层句柄。调用者不能：

- 对 `get()` 的结果调用 `std::fclose()`；
- 把该指针保存到 `FileGuard` 生命周期之外；
- 假设借用指针可以独立于拥有者存在。

这延续了第一部分的概念：拥有者决定生命周期，借用者只能在拥有者有效时访问资源。

下面用同一个包装器重新打开文件，并通过借用指针调用 `std::fgets()`：

In [ ]:
// 本步骤：通过代码演示“本节实验”并观察结果。
{
    FileGuard file(raii_file_path, "r");
    char buffer[128]{};

    if (std::fgets(buffer, sizeof(buffer), file.get()) != nullptr)
    {
        std::cout << "content: " << buffer << std::endl;
    }
}

### 6. 获取资源失败

如果文件打开失败，构造函数抛出异常，`FileGuard` 对象不会进入有效生命周期。由于 `std::fopen()` 没有成功返回资源，此时也没有文件句柄需要析构函数释放。

In [ ]:
// 本步骤：通过代码演示“获取资源失败”并观察结果。
try
{
    FileGuard missing(
        "/path/that/does/not/exist/input.txt",
        "r");
}
catch (const std::exception &error)
{
    std::cout << "open failed: " << error.what() << std::endl;
}

### 7. 为什么禁止复制

默认复制会让两个对象保存同一个 `std::FILE*`，两个析构函数随后会重复关闭同一资源：

```cpp
FileGuard first(path, "w");
FileGuard second = first; // 已被 = delete 禁止
```

`= delete` 把非法所有权操作变成编译错误。后续学习移动语义时，再允许资源所有权从一个对象安全转移到另一个对象。

### 8. 如何理解 Resource Acquisition Is Initialization

字面翻译是“资源获取即初始化”，更实用的理解是：

> 如果一个对象代表某项资源，那么对象的合法生命周期就应同时代表资源的合法生命周期。

构造成功意味着对象拥有有效资源；对象存在期间资源保持有效；对象析构意味着资源被释放。调用者不再单独维护第二套资源生命周期。

### 9. 手动管理与 RAII 对比

| 问题 | 手动 `fopen/fclose` | `FileGuard` |
| --- | --- | --- |
| 谁拥有句柄 | 依靠约定 | 对象明确拥有 |
| 正常返回 | 手动关闭 | 自动析构关闭 |
| 提前返回 | 每个分支都要关闭 | 自动析构关闭 |
| 重复释放 | 依靠人工避免 | 禁止复制 |
| 打开失败 | 手动传播错误 | 构造失败，不产生无效对象 |

析构函数通常不应抛出异常。本实验忽略 `std::fclose()` 的返回值；生产代码若需要报告刷新或关闭失败，可以额外设计显式操作，但析构函数仍应提供不抛异常的兜底清理。

### 实验结论

RAII 把“必须记得调用 `std::fclose()`”转换为“局部对象离开作用域时必然析构”。资源释放从分散的控制流约定变成类型自身的行为。

后续实验会继续观察嵌套作用域、多个资源的析构顺序、异常栈展开，以及更完整的 `File` 类。